<a href="https://colab.research.google.com/github/Akashkumar12/AI-ML/blob/main/Hackathon_U2_MH1_AuthorIdentification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

## Problem Statement

The problem is to identify the author of a  book from a given list of possible authors.

## Learning Objectives

At the end of the experiment, you will be able to:

* Use NLTK package
* Extract handcrafted features
* Preprocess the text
* Write an algorithm to identify the author of a given book


In [ ]:
#@title  Mini Hackathon Walkthrough
from IPython.display import HTML

HTML("""<video width="854" height="480" controls>
  <source src="https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Walkthrough/authoridentification.mp4" type="video/mp4">
</video>
""")

## Background

Author identification is the task of identifying the author of a given text. It can be considered as a typical classification problem, where a set of books with known authors are used for training. The aim is to automatically determine the corresponding author of an anonymous text.

## Grading = 10 Marks

## Setup Steps

In [ ]:
#@title Run this cell to complete the setup for this Notebook

from IPython import get_ipython
ipython = get_ipython()

notebook="U2_MH1_AuthorIdentification" #name of the notebook
Answer = "This notebook is graded by mentors on the day of hackathon"
def setup():
    ipython.magic("sx wget https://cdn.talentsprint.com/talentsprint1/archives/sc/aiml/experiment_related_data/AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD.rar")
    ipython.magic("sx unrar e /content/AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD.rar")
    print ("Setup completed successfully")
    return

setup()

Setup completed successfully


### NOTE: You are allowed to use ML libraries such as Sklearn, NLTK, etc wherever applicable

### Downloading the required nltk Packages before moving ahead

In [ ]:
import nltk
nltk.download('gutenberg')
nltk.download('punkt')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## **Stage 1:** Dataset Preparation

### 1 Marks -> Ensure you appropriately split the multiple short stories for the below-mentioned authors, Which will be your training data.

**1.** Before moving ahead choose two authors based on your team-number allocation: <br/>


Team=1,5,9,13,17,21  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;    Author-A Vs Author-B <br />
Team=2,6,10,14,18,22 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;         Author-B Vs Author-C <br />
Team=3,7,11,15,19,23 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;         Author-C Vs Author-D <br />
Team=4,8,12,16,20,24 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;           Author-D Vs Author-E <br />



**2.** Link to the short stories collection of each author for your problem: <br />

*   Author-A -> Rudyard Kipling   [Short Stories Collection](http://www.gutenberg.org/files/2781/2781-0.txt) &nbsp;&nbsp;
*   Author-B -> Anton Chekhov [Short Stories Collection](http://www.gutenberg.org/files/1732/1732-0.txt) &nbsp;&nbsp;
*   Author-C -> Guy De Maupassant [Short Stories Collection](http://www.gutenberg.org/cache/epub/21327/pg21327.txt)&nbsp;&nbsp;
*   Author-D -> Mark Twain [Short Stories Collection](http://www.gutenberg.org/files/245/245-0.txt)&nbsp;&nbsp;
*   Author-E -> Saki [Short Stories Collection](http://www.gutenberg.org/files/1477/1477-0.txt)&nbsp;&nbsp;

**Hint for downloading raw text from Gutenberg :**  Refer to the section "Electronic Books" in the following  [link](https://www.nltk.org/book/ch03.html) for the instructions.



**Hint for finding the index of a text:**   You may use `raw.find()` and `raw.rfind()` in the same [link](https://www.nltk.org/book/ch03.html) to find the appropriate index of the start and end location

**Hint for splitting the multiple stories:** Split the stories using long space (white space character)

**Note:** Ignore the table of contents section from the given stories

In [ ]:
import requests
import re
import pandas as pd

# URLs for the short stories
CHEKHOV_URL = "http://www.gutenberg.org/files/1732/1732-0.txt"
MAUPASSANT_URL = "http://www.gutenberg.org/cache/epub/21327/pg21327.txt"

def download_and_clean_gutenberg_text(url):
    """Downloads text from a Gutenberg URL and removes header/footer."""
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for HTTP errors
    raw_text = response.text

    # Find the start and end markers of the actual book content
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

    start_index = raw_text.find(start_marker)
    end_index = raw_text.rfind(end_marker)

    if start_index == -1 or end_index == -1:

        if "ANTON CHEKHOV" in url:

            content_start_search = re.search(r'\n\s*I\n\s*At first there was nothing, only the river.', raw_text)
            if content_start_search:
                start_index = content_start_search.start()
                # Find the last double newline before this specific content to get closer to the actual start of stories
                temp_start_index = raw_text.rfind('\n\n', 0, start_index)
                if temp_start_index != -1:
                    start_index = temp_start_index
                else: # Fallback if no double newline found before
                    start_index = 0
            else:
                start_index = 0
            end_index = len(raw_text) # Keep everything till the end if end marker not found

        elif "GUY DE MAUPASSANT" in url.upper():

            content_start_search = re.search(r'\n\s*I\.\n\n\s*Mademoiselle Fifi', raw_text)
            if content_start_search:
                start_index = content_start_search.start()
                # Find the last double newline before this specific content to get closer to the actual start of stories
                temp_start_index = raw_text.rfind('\n\n', 0, start_index)
                if temp_start_index != -1:
                    start_index = temp_start_index
                else: # Fallback if no double newline found before
                    start_index = 0
            else:
                start_index = 0
            end_index = len(raw_text) # Keep everything till the end if end marker not found
        else:
            # Default to no cleaning for unhandled cases
            start_index = 0
            end_index = len(raw_text)

    else:
        # Adjust start_index to be after the marker
        start_index += len(start_marker)

    cleaned_text = raw_text[start_index:end_index]
    return cleaned_text.strip()

def split_stories(text, min_length=200):

    story_splits = re.split(r'\n\s*\n\s*\n\s*\n+', text)

    stories = [s.strip() for s in story_splits if len(s.strip()) > min_length]
    return stories

# --- Download and process Anton Chekhov's stories ---
print("Processing Anton Chekhov's stories...")
chekhov_raw_text = download_and_clean_gutenberg_text(CHEKHOV_URL)
chekhov_stories = split_stories(chekhov_raw_text)
print(f"Found {len(chekhov_stories)} stories for Anton Chekhov.")

# --- Download and process Guy De Maupassant's stories ---
print("\nProcessing Guy De Maupassant's stories...")
maupassant_raw_text = download_and_clean_gutenberg_text(MAUPASSANT_URL)
maupassant_stories = split_stories(maupassant_raw_text)
print(f"Found {len(maupassant_stories)} stories for Guy De Maupassant.")

# Create a combined dataset for training (example structure)
all_stories = []
for story in chekhov_stories:
    all_stories.append({'story': story, 'author': 'Anton Chekhov'})
for story in maupassant_stories:
    all_stories.append({'story': story, 'author': 'Guy De Maupassant'})

print(f"\nTotal stories collected: {len(all_stories)}")


df_stories = pd.DataFrame(all_stories)
display(df_stories.head())
display(df_stories['author'].value_counts())

Processing Anton Chekhov's stories...
Found 25 stories for Anton Chekhov.

Processing Guy De Maupassant's stories...
Found 31 stories for Guy De Maupassant.

Total stories collected: 56


,story,author
0,﻿Project Gutenberg’s The Schoolmistress and Ot...,Anton Chekhov
1,"FROM THE TALES OF CHEKHOV, VOLUME 9\r\n\r\nCON...",Anton Chekhov
2,THE SCHOOLMISTRESS\r\n\r\nAT half-past eight t...,Anton Chekhov
3,A NERVOUS BREAKDOWN\r\n\r\nA MEDICAL student c...,Anton Chekhov
4,MISERY\r\n\r\n“To whom shall I tell my grief?”...,Anton Chekhov


,count
author,
Guy De Maupassant,31
Anton Chekhov,25


In [ ]:
display(df_stories.head())

,story,author
0,﻿Project Gutenberg’s The Schoolmistress and Ot...,Anton Chekhov
1,"FROM THE TALES OF CHEKHOV, VOLUME 9\r\n\r\nCON...",Anton Chekhov
2,THE SCHOOLMISTRESS\r\n\r\nAT half-past eight t...,Anton Chekhov
3,A NERVOUS BREAKDOWN\r\n\r\nA MEDICAL student c...,Anton Chekhov
4,MISERY\r\n\r\n“To whom shall I tell my grief?”...,Anton Chekhov


In [ ]:
df_stories.columns


Index(['story', 'author'], dtype='object')

## **Stage 2**: Experiment with Handcrafted features representation
Extract Handcrafted features for the obtained short stories from **Stage-1**

**Stylometry:**

Each person has a unique vocabulary, sometimes rich, sometimes limited. Although a larger vocabulary is usually associated with literary quality, this is not always the case. Ernest Hemingway is famous for using a surprisingly small number of different words in his writing, which did not prevent him from winning the Nobel Prize for Literature in 1954.

Some people write in short sentences, while others prefer long blocks of text consisting of many clauses. No two people use semicolons, em-dashes, and other forms of punctuation in the same way.




**You may explore the following ways to analyze the text and generate handcrafted features by searching text in a probing way:**

a)  Could the style of punctuation usage help as a handcrafted feature? Both by those who follow punctuations and by those who don't? Interesting [link](https://qwiklit.com/2014/03/05/top-10-authors-who-ignored-the-basic-rules-of-punctuation/)

b) The same word can sometimes be used in different contexts repeatedly by different authors. Could this fact be converted as a handcrafted feature? [link](https://www.nltk.org/book/ch01.html)

c) The above two are merely examples; As you might have noticed already the NLTK book [link](https://www.nltk.org/book/) offers several methods of analyzing and understanding the text. Each of these analyses is in itself capable of being a handcrafted feature. **However for your evaluation a minimal set of useful handcrafted features which is helping you prove a classification of an is sufficient**

d) Could most common words be used to distinguish authors?  Refer "Counting Vocabulary" section of the [link](https://www.nltk.org/book/ch01.html)

e) How about using a count of most frequently used bi-gram, tri-grams, and using it to classify an author?

f) How about using the frequency histogram of the most frequently used words across the stories by a given author a useful feature?

The limit here is endlessly limited only by your imagination, and of course your accuracy! :)


### 1 Marks ->  a) List 6 handcrafted features to distinguish author stories.

In [ ]:
# For eg:
# 1. UniqueWords
# 2. AvgSentLength
# List the other handcrafted features here

In [ ]:
# story
# author
# unique_words
# avg_sentence_length
# avg_word_length
# punctuation_frequency

###  2 Marks -> b) Write functions for any 4 of the above 6 handcrafted features and label your authors accordingly.

- Get any 4 hand crafted features from the above listed 6 hand-crafted features for every story obtained from **stage-1**.
- Identify your target variable as an author and label them accordingly.

In [ ]:
# Stories_list    UniqueWords    AvgSentLength     Label
#     1               x1               x2            y

# YOUR CODE HERE

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
nltk.download('punkt_tab')

# Ensure NLTK resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

def calculate_unique_words(text):
    words = word_tokenize(text.lower())
    words = [word for word in words if word.isalpha()] # Filter out punctuation and numbers
    return len(set(words))

def calculate_avg_sentence_length(text):
    sentences = sent_tokenize(text)
    if not sentences: return 0
    total_words = sum(len(word_tokenize(sentence)) for sentence in sentences)
    return total_words / len(sentences)

def calculate_avg_word_length(text):
    words = word_tokenize(text.lower())
    words = [word for word in words if word.isalpha()] # Filter out punctuation and numbers
    if not words: return 0
    total_chars = sum(len(word) for word in words)
    return total_chars / len(words)

def calculate_punctuation_frequency(text, punctuation_marks=['.', ',', '!', '?', ';', ':']):
    total_chars = len(text)
    if total_chars == 0: return 0
    punctuation_count = sum(text.count(p) for p in punctuation_marks)
    return punctuation_count / total_chars

# Apply the feature extraction functions to the DataFrame
df_stories['unique_words'] = df_stories['story'].apply(calculate_unique_words)
df_stories['avg_sentence_length'] = df_stories['story'].apply(calculate_avg_sentence_length)
df_stories['avg_word_length'] = df_stories['story'].apply(calculate_avg_word_length)
df_stories['punctuation_frequency'] = df_stories['story'].apply(calculate_punctuation_frequency)

# Display the DataFrame with new features and value counts for the author column
display(df_stories.head())
display(df_stories['author'].value_counts())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,story,author,unique_words,avg_sentence_length,avg_word_length,punctuation_frequency
0,﻿Project Gutenberg’s The Schoolmistress and Ot...,Anton Chekhov,52,56.500000,5.132530,0.021002
1,"FROM THE TALES OF CHEKHOV, VOLUME 9\r\n\r\nCON...",Anton Chekhov,38,63.000000,4.727273,0.001996
2,THE SCHOOLMISTRESS\r\n\r\nAT half-past eight t...,Anton Chekhov,952,27.724832,4.268635,0.029048
3,A NERVOUS BREAKDOWN\r\n\r\nA MEDICAL student c...,Anton Chekhov,1793,24.557604,4.269947,0.031671
4,MISERY\r\n\r\n“To whom shall I tell my grief?”...,Anton Chekhov,672,18.821429,3.975645,0.047905


,count
author,
Guy De Maupassant,31
Anton Chekhov,25


##**Stage 3:** Experiment with Text processing and representation:
Extract features using TFIDF or CountVectorizer or Word2vec for the obtained short stories from **Stage-1**



### 1 Mark -> a) Performing basic cleanup operations such as removing the newline characters and removing trailing spaces

**For example,** Your sentence looks as follows \[' This is a sentence\n\r. Another sentence \n'].

After newline removal from the above example, your sentence will look like \['This is a sentence. Another sentence'].

 In order to do this, you can try using a combination of split() and join()

In [ ]:
print(df_stories['story'].head())
df_stories['story'] = df_stories['story'].apply(lambda x: x.replace('\n', ' ').replace('\r', ' ').strip())
print(df_stories['story'].head())



0    ﻿Project Gutenberg’s The Schoolmistress and Ot...
1    FROM THE TALES OF CHEKHOV, VOLUME 9\r\n\r\nCON...
2    THE SCHOOLMISTRESS\r\n\r\nAT half-past eight t...
3    A NERVOUS BREAKDOWN\r\n\r\nA MEDICAL student c...
4    MISERY\r\n\r\n“To whom shall I tell my grief?”...
Name: story, dtype: object
0    ﻿Project Gutenberg’s The Schoolmistress and Ot...
1    FROM THE TALES OF CHEKHOV, VOLUME 9    CONTENT...
2    THE SCHOOLMISTRESS    AT half-past eight they ...
3    A NERVOUS BREAKDOWN    A MEDICAL student calle...
4    MISERY    “To whom shall I tell my grief?”    ...
Name: story, dtype: object


###  2 Marks-> b) Generate vectors for the given stories

Create a representation of text, convert it into vectors (numbers)


**Use any one** of the following algorithms for this task :

* Countvectorizer or
* TFIDFVectorizer or
* Word2Vec (The word2vec bin file (AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD) can be downloaded as a part of setup  )
  * perform sentence level tokenization and word level tokenization for the given stories

    **Example of sentences as list of words:**<br/>
    **Before:** ['This is a sentence .' , ' Another sentence']<br/>
    **After:** ['This', 'is' ,'a', 'sentence' , ' . ' , ' Another ', ' sentence ' ]
 * Assign the respective label associated for each vector representation of the extracted word

References Documents:

1.   [Countvectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
2.  [TFIDFVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)


In [ ]:
import sys
!{sys.executable} -m pip install gensim

In [ ]:
from gensim.models import KeyedVectors
import numpy as np
from nltk.tokenize import word_tokenize

# Load the pre-trained Google News Word2Vec model
# The path is based on the setup cell which unrar'd the file to /content/
print("Loading Word2Vec model... This might take a few minutes.")
word2vec_model = KeyedVectors.load_word2vec_format('/content/AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD.bin', binary=True)
print("Word2Vec model loaded successfully.")

def get_story_vector(text, model, vector_size=300):
    """Converts a story into a vector by averaging word embeddings."""
    words = word_tokenize(text.lower()) # Tokenize words

    # Filter out words not in the Word2Vec model's vocabulary
    word_vectors = [model[word] for word in words if word in model.key_to_index]

    if not word_vectors:
        return np.zeros(vector_size) # Return a zero vector if no words found in vocabulary
    else:
        return np.mean(word_vectors, axis=0)

# Apply the function to create story vectors
print("Generating story vectors...")
df_stories['word2vec_vector'] = df_stories['story'].apply(lambda x: get_story_vector(x, word2vec_model))

display(df_stories[['story', 'word2vec_vector']].head())

Loading Word2Vec model... This might take a few minutes.
Word2Vec model loaded successfully.
Generating story vectors...


,story,word2vec_vector
0,﻿Project Gutenberg’s The Schoolmistress and Ot...,"[0.039881, -0.035777044, 0.011560978, 0.066459..."
1,"FROM THE TALES OF CHEKHOV, VOLUME 9 CONTENT...","[0.081030525, 0.04562523, 0.04033484, 0.066405..."
2,THE SCHOOLMISTRESS AT half-past eight they ...,"[0.058667123, 0.040267512, 0.03388163, 0.06693..."
3,A NERVOUS BREAKDOWN A MEDICAL student calle...,"[0.059933078, 0.0444696, 0.043897185, 0.077362..."
4,MISERY “To whom shall I tell my grief?” ...,"[0.062992595, 0.044828124, 0.030377546, 0.0795..."


In [ ]:
# YOUR CODE HERE (HINT: Convert to numpy array if needed)

###  1 Mark -> c) Is stop word removal necessary in the context of author identification? Your thoughts below?

In [ ]:
# YOUR ANSWER IN TEXT
# No Need

##**Stage 4:** Classification :

### Expected accuracy is above 85%

### 2 Marks -> Perform a classification using either features obtained from Stage2 or Stage3

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Prepare features (X) and labels (y)
X = np.array(df_stories['word2vec_vector'].tolist())

# Encode author labels to numerical values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_stories['author'])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 44
Test set size: 12


In [ ]:
# Initialize and train a RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")


Model Accuracy: 0.7500


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf_vectorizer.fit_transform(df_stories['story']).toarray()

label_encoder = LabelEncoder()
y_tfidf = label_encoder.fit_transform(df_stories['author'])

X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf, y_tfidf, test_size=0.2, random_state=42)

print(f"Training set size (TF-IDF features): {X_train_tfidf.shape[0]}")
print(f"Test set size (TF-IDF features): {X_test_tfidf.shape[0]}")
print(f"Number of TF-IDF features: {X_tfidf.shape[1]}")

Training set size (TF-IDF features): 44
Test set size (TF-IDF features): 12
Number of TF-IDF features: 1000


In [ ]:

classifier_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_tfidf.fit(X_train_tfidf, y_train_tfidf)

y_pred_tfidf = classifier_tfidf.predict(X_test_tfidf)

accuracy_tfidf = accuracy_score(y_test_tfidf, y_pred_tfidf)
print(f"Model Accuracy (TF-IDF Features): {accuracy_tfidf:.4f}")



Model Accuracy (TF-IDF Features): 0.9167


## Classification using CountVectorizer Features (Stage 3)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Initialize CountVectorizer
count_vectorizer = CountVectorizer(max_features=1000) # Limiting features for manageability

# Fit and transform the 'story' column to get CountVectorizer features
X_cv = count_vectorizer.fit_transform(df_stories['story']).toarray()

# Use the same encoded author labels
label_encoder = LabelEncoder()
y_cv = label_encoder.fit_transform(df_stories['author'])

# Split data into training and testing sets for CountVectorizer features
X_train_cv, X_test_cv, y_train_cv, y_test_cv = train_test_split(X_cv, y_cv, test_size=0.2, random_state=42)

print(f"Training set size (CountVectorizer features): {X_train_cv.shape[0]}")
print(f"Test set size (CountVectorizer features): {X_test_cv.shape[0]}")
print(f"Number of CountVectorizer features: {X_cv.shape[1]}")

Training set size (CountVectorizer features): 44
Test set size (CountVectorizer features): 12
Number of CountVectorizer features: 1000


In [ ]:
# Initialize and train a RandomForestClassifier with CountVectorizer features
classifier_cv = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_cv.fit(X_train_cv, y_train_cv)

# Make predictions on the test set
y_pred_cv = classifier_cv.predict(X_test_cv)

# Evaluate the model
accuracy_cv = accuracy_score(y_test_cv, y_pred_cv)
print(f"Model Accuracy (CountVectorizer Features): {accuracy_cv:.4f}")

# print("\nClassification Report (CountVectorizer Features):")
# print(classification_report(y_test_cv, y_pred_cv, target_names=label_encoder.classes_))

Model Accuracy (CountVectorizer Features): 0.7500


# Further Ideas for exploration after the hackathon:

**Statistical analysis** of text using NLP, by analysis meaning of sentences, feature based grammars and analyzing structure of sentences!

reference: www.nltk.org/book